# 03 生产环境最佳实践 (Production Best Practices)

## 1. 概述

### 学习目标

完成本notebook后，你将能够：

1. **模型版本管理** - 使用语义版本控制和模型注册表管理模型生命周期
2. **A/B测试** - 设计和实施A/B测试，科学地比较模型版本
3. **金丝雀部署** - 实现渐进式发布，降低部署风险
4. **监控与漂移检测** - 检测数据漂移和概念漂移，保障模型质量
5. **错误处理** - 构建健壮的错误处理和优雅降级机制
6. **SLA管理** - 定义和监控服务级别协议
7. **事故响应** - 建立事故响应流程和复盘机制

### 核心概念速览

**数据漂移 (Data Drift)** 是指模型输入数据的统计分布随时间发生变化的现象，例如用户行为模式改变或数据采集方式变动导致特征分布偏移。在本章场景中，它是模型监控的核心指标，用于判断模型是否需要重新训练。

**概念漂移 (Concept Drift)** 是指输入特征与目标变量之间的映射关系随时间发生变化的现象，即同样的输入在不同时期对应不同的正确输出。在本章场景中，它比数据漂移更难检测，通常需要通过监控模型预测准确率的下降来间接发现。


### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 线上出现不安全回复如何紧急处理？→ 安全护栏与内容审核机制
- 模型效果随时间漂移怎么发现？→ 监控告警与持续评估体系

### 前置要求

- 完成 01_inference_optimization.ipynb（推理优化）
- 完成 02_model_serving.ipynb（模型服务）
- 理解 REST API 和微服务架构

### 内容结构

```
Section 2: MLOps基础 .............. MLOps生命周期、研究vs生产
Section 3: 模型版本管理 ........... 实验追踪、模型注册表
Section 4: A/B测试与金丝雀部署 .... 流量分割、渐进发布
Section 5: 监控与漂移检测 ......... 数据漂移、概念漂移、告警
Section 6: 错误处理与调试 ......... 输入验证、优雅降级、日志
Section 7: SLA与性能优化 .......... SLA/SLO/SLI、性能调优
Section 8: 事故响应与复盘 ......... 事故管理、复盘模板
```

> ⏱️ **预计时间**: 2-3小时 | 📊 **难度**: ⭐⭐⭐⭐ 高级

In [ ]:
# === Production Best Practices - Setup ===
import numpy as np
import json
import time
import hashlib
from datetime import datetime, timedelta
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print(f"  NumPy: {np.__version__}")
print(f"  Python datetime, json, hashlib ready")
print(f"\n📌 This notebook covers MLOps best practices:")
print(f"   - Model versioning and registry")
print(f"   - A/B testing and canary deployment")
print(f"   - Monitoring and drift detection")
print(f"   - Error handling and debugging")
print(f"   - SLA monitoring and optimization")
print(f"   - Incident response and postmortem")

## 2. MLOps基础

### 2.1 MLOps生命周期

```
开发 (Development)
  ↓ 数据准备、特征工程、模型选择
训练 (Training)
  ↓ 超参调优、交叉验证
验证 (Validation)
  ↓ 离线评估、A/B测试
部署 (Deployment)
  ↓ 模型服务、API发布
监控 (Monitoring)
  ↓ 性能追踪、漂移检测
迭代 (Iteration)
  ↓ 重训练、模型更新
  → 回到开发阶段
```

### 2.2 研究 vs 生产

| 维度 | 研究环境 | 生产环境 |
|------|---------|---------|
| **目标** | 最高精度 | 可靠性 + 合理精度 |
| **数据** | 固定数据集 | 持续变化的数据流 |
| **评估** | 离线指标 | 在线指标 + 业务指标 |
| **延迟** | 不重要 | 关键约束 |
| **成本** | 不考虑 | 核心考量 |
| **错误处理** | 手动修复 | 自动恢复 |
| **可复现性** | 尽力而为 | 严格要求 |
| **监控** | TensorBoard | Prometheus + Grafana |

### 2.3 MLOps成熟度模型

| 级别 | 描述 | 特征 |
|------|------|------|
| **Level 0** | 手动流程 | 手动训练、手动部署、无监控 |
| **Level 1** | ML流水线 | 自动训练、手动部署、基础监控 |
| **Level 2** | CI/CD | 自动训练+部署、A/B测试、完整监控 |
| **Level 3** | 全自动 | 自动重训练、自动回滚、自适应系统 |

> **目标：** 本notebook覆盖Level 2-3的核心实践

In [ ]:
# 🔬 Micro Practice: Production Readiness Checklist
# Goal: Build a comprehensive checklist for deploying ML models to production
# Expected outcome: Systematic evaluation of production readiness

class ProductionReadinessChecker:
    """Comprehensive production readiness assessment"""
    
    def __init__(self, model_name, version):
        self.model_name = model_name
        self.version = version
        self.checks = {}
        self.results = {}
        self._setup_checks()
        print(f"✓ Production Readiness Checker for {model_name} v{version}")
    
    def _setup_checks(self):
        """Define all production readiness checks"""
        self.checks = {
            "model_quality": {
                "description": "模型质量验证",
                "items": [
                    ("accuracy_threshold", "模型精度达到阈值 (>90%)"),
                    ("no_data_leakage", "无数据泄露"),
                    ("bias_check", "偏差检测通过"),
                    ("edge_cases", "边界情况测试通过"),
                    ("regression_test", "回归测试通过"),
                ]
            },
            "infrastructure": {
                "description": "基础设施就绪",
                "items": [
                    ("latency_sla", "延迟满足SLA (<100ms P95)"),
                    ("throughput_sla", "吞吐量满足需求 (>100 QPS)"),
                    ("memory_budget", "内存在预算内 (<4GB)"),
                    ("gpu_utilization", "GPU利用率合理 (60-80%)"),
                    ("auto_scaling", "自动扩缩容配置"),
                ]
            },
            "reliability": {
                "description": "可靠性保障",
                "items": [
                    ("health_check", "健康检查端点"),
                    ("graceful_degradation", "优雅降级策略"),
                    ("rollback_plan", "回滚方案就绪"),
                    ("circuit_breaker", "熔断器配置"),
                    ("retry_policy", "重试策略配置"),
                ]
            },
            "monitoring": {
                "description": "监控与告警",
                "items": [
                    ("metrics_dashboard", "指标仪表板"),
                    ("alerting_rules", "告警规则配置"),
                    ("logging", "结构化日志"),
                    ("drift_detection", "漂移检测"),
                    ("error_tracking", "错误追踪"),
                ]
            },
            "security": {
                "description": "安全与合规",
                "items": [
                    ("input_validation", "输入验证"),
                    ("rate_limiting", "速率限制"),
                    ("authentication", "认证授权"),
                    ("data_privacy", "数据隐私合规"),
                    ("audit_logging", "审计日志"),
                ]
            },
            "documentation": {
                "description": "文档与运维",
                "items": [
                    ("api_docs", "API文档完整"),
                    ("runbook", "运维手册"),
                    ("incident_playbook", "事故响应手册"),
                    ("model_card", "模型卡片"),
                    ("changelog", "变更日志"),
                ]
            }
        }
    
    def run_check(self, category, item_key, passed, notes=""):
        """Run a specific check"""
        if category not in self.results:
            self.results[category] = {}
        self.results[category][item_key] = {
            "passed": passed,
            "notes": notes
        }
    
    def run_all_simulated(self):
        """Simulate running all checks with realistic results"""
        import random
        random.seed(42)
        
        for category, info in self.checks.items():
            for item_key, description in info["items"]:
                # Simulate: 85% pass rate
                passed = random.random() < 0.85
                notes = "Auto-verified" if passed else "Needs attention"
                self.run_check(category, item_key, passed, notes)
    
    def generate_report(self):
        """Generate production readiness report"""
        print(f"\n{'='*60}")
        print(f"📋 Production Readiness Report")
        print(f"   Model: {self.model_name} v{self.version}")
        print(f"{'='*60}")
        
        total_passed = 0
        total_checks = 0
        category_scores = {}
        
        for category, info in self.checks.items():
            cat_passed = 0
            cat_total = len(info["items"])
            
            print(f"\n📌 {info['description']} ({category})")
            print(f"   {'Check':<35} {'Status':<10} {'Notes'}")
            print(f"   {'-'*65}")
            
            for item_key, description in info["items"]:
                result = self.results.get(category, {}).get(item_key, {"passed": False, "notes": "Not run"})
                status = "✅ PASS" if result["passed"] else "❌ FAIL"
                print(f"   {description:<35} {status:<10} {result['notes']}")
                
                if result["passed"]:
                    cat_passed += 1
                total_checks += 1
            
            total_passed += cat_passed
            score = cat_passed / cat_total * 100
            category_scores[category] = score
            print(f"   Score: {cat_passed}/{cat_total} ({score:.0f}%)")
        
        # Overall summary
        overall_score = total_passed / total_checks * 100
        print(f"\n{'='*60}")
        print(f"📊 Overall Score: {total_passed}/{total_checks} ({overall_score:.0f}%)")
        
        if overall_score >= 95:
            print("🟢 READY FOR PRODUCTION")
        elif overall_score >= 80:
            print("🟡 CONDITIONAL APPROVAL - Fix failing checks")
        else:
            print("🔴 NOT READY - Significant issues to resolve")
        
        # Show failing items
        failing = []
        for category, info in self.checks.items():
            for item_key, description in info["items"]:
                result = self.results.get(category, {}).get(item_key, {"passed": False})
                if not result["passed"]:
                    failing.append(f"  - [{category}] {description}")
        
        if failing:
            print(f"\n⚠️  Action Items ({len(failing)} items):")
            for item in failing:
                print(item)
        
        print(f"{'='*60}")
        return overall_score, category_scores


# === Demo ===
checker = ProductionReadinessChecker("sentiment-model", "1.1.0")
checker.run_all_simulated()
overall_score, category_scores = checker.generate_report()

print(f"\n✅ Production readiness assessment complete!")

## 3. 模型版本管理与注册

### 3.1 为什么需要版本管理？

| 问题 | 没有版本管理 | 有版本管理 |
|------|-------------|-----------|
| **可复现性** | "上周的模型在哪？" | 精确追踪每个版本 |
| **回滚** | 无法恢复到之前版本 | 一键回滚 |
| **协作** | 团队成员覆盖彼此工作 | 并行实验，清晰对比 |
| **审计** | 不知道谁改了什么 | 完整变更历史 |

### 3.2 语义版本控制

```
MAJOR.MINOR.PATCH

MAJOR: 架构变更（如 BERT → GPT）
MINOR: 训练改进（如 新数据、超参调优）
PATCH: Bug修复（如 预处理修复）

示例: sentiment-model v2.1.3
  2 = 第二代架构 (DistilBERT)
  1 = 第一次训练改进
  3 = 第三次Bug修复
```

### 3.3 实验追踪 (MLflow风格)

需要追踪的内容：

| 类别 | 内容 | 示例 |
|------|------|------|
| **参数** | 超参数、配置 | lr=2e-5, epochs=3 |
| **指标** | 训练/验证指标 | accuracy=0.91, loss=0.22 |
| **工件** | 模型文件、图表 | model.pt, loss_curve.png |
| **环境** | 代码版本、依赖 | git_hash, requirements.txt |
| **数据** | 数据集版本 | dataset_v2.1, hash=abc123 |

### 3.4 模型注册表

```
模型生命周期:

None → Staging → Production → Archived
  ↑       ↑          ↑           ↑
 注册   验证通过    上线发布    新版本替代

关键规则:
- 同一时间只有一个 Production 版本
- 升级到 Production 时自动归档旧版本
- Staging 需要通过所有验证测试
```

In [ ]:
# 🔬 Micro Practice: Model Versioning and Registry
# Goal: Implement experiment tracking and model registry
# Expected outcome: Track experiments, manage model versions, promote models through stages

import hashlib
from datetime import datetime

class ExperimentTracker:
    """MLflow-style experiment tracking"""
    
    def __init__(self, experiment_name):
        self.experiment_name = experiment_name
        self.runs = []
        self.current_run = None
        print(f"✓ Experiment '{experiment_name}' created")
    
    def start_run(self, run_name=None):
        """Start a new experiment run"""
        run_id = hashlib.md5(f"{datetime.now()}{run_name}".encode()).hexdigest()[:8]
        self.current_run = {
            'run_id': run_id,
            'run_name': run_name or f"run_{run_id}",
            'start_time': datetime.now().isoformat(),
            'parameters': {},
            'metrics': {},
            'artifacts': [],
            'tags': {},
            'status': 'RUNNING'
        }
        print(f"  Started run: {self.current_run['run_name']} (ID: {run_id})")
        return run_id
    
    def log_param(self, key, value):
        """Log a parameter"""
        if self.current_run is None:
            raise RuntimeError("No active run")
        self.current_run['parameters'][key] = value
    
    def log_params(self, params):
        """Log multiple parameters"""
        for k, v in params.items():
            self.log_param(k, v)
    
    def log_metric(self, key, value, step=None):
        """Log a metric"""
        if self.current_run is None:
            raise RuntimeError("No active run")
        if key not in self.current_run['metrics']:
            self.current_run['metrics'][key] = []
        self.current_run['metrics'][key].append({
            'value': value,
            'step': step,
            'timestamp': datetime.now().isoformat()
        })
    
    def log_artifact(self, artifact_path):
        """Log an artifact (model file, plot, etc.)"""
        if self.current_run is None:
            raise RuntimeError("No active run")
        self.current_run['artifacts'].append(artifact_path)
    
    def end_run(self, status='COMPLETED'):
        """End the current run"""
        if self.current_run is None:
            raise RuntimeError("No active run")
        self.current_run['end_time'] = datetime.now().isoformat()
        self.current_run['status'] = status
        self.runs.append(self.current_run)
        print(f"  Completed run: {self.current_run['run_name']} ({status})")
        self.current_run = None
    
    def get_best_run(self, metric, mode='max'):
        """Get the best run based on a metric"""
        best_run = None
        best_value = float('-inf') if mode == 'max' else float('inf')
        
        for run in self.runs:
            if metric in run['metrics']:
                values = [m['value'] for m in run['metrics'][metric]]
                final_value = values[-1]  # Last logged value
                if mode == 'max' and final_value > best_value:
                    best_value = final_value
                    best_run = run
                elif mode == 'min' and final_value < best_value:
                    best_value = final_value
                    best_run = run
        
        return best_run, best_value
    
    def compare_runs(self):
        """Compare all runs"""
        print(f"\n📊 Experiment: {self.experiment_name}")
        print(f"{'Run Name':<20} {'Status':<12} {'Parameters':<30} {'Metrics'}")
        print("-" * 90)
        
        for run in self.runs:
            params_str = str({k: v for k, v in list(run['parameters'].items())[:3]})[:28]
            metrics_str = {k: round(v[-1]['value'], 4) for k, v in run['metrics'].items()}
            print(f"{run['run_name']:<20} {run['status']:<12} {params_str:<30} {metrics_str}")


class ModelRegistry:
    """Model registry with staging/production management"""
    
    STAGES = ['None', 'Staging', 'Production', 'Archived']
    
    def __init__(self):
        self.models = {}  # name -> list of versions
        print("✓ Model Registry initialized")
    
    def register_model(self, name, version, run_id, metrics=None, description=""):
        """Register a new model version"""
        if name not in self.models:
            self.models[name] = []
        
        model_version = {
            'name': name,
            'version': version,
            'run_id': run_id,
            'metrics': metrics or {},
            'description': description,
            'stage': 'None',
            'created_at': datetime.now().isoformat(),
            'updated_at': datetime.now().isoformat()
        }
        
        self.models[name].append(model_version)
        print(f"  Registered: {name} v{version} (run: {run_id})")
        return model_version
    
    def transition_stage(self, name, version, new_stage):
        """Transition model to a new stage"""
        if new_stage not in self.STAGES:
            raise ValueError(f"Invalid stage: {new_stage}. Must be one of {self.STAGES}")
        
        model = self._get_version(name, version)
        if model is None:
            raise ValueError(f"Model {name} v{version} not found")
        
        old_stage = model['stage']
        
        # If promoting to Production, archive current production model
        if new_stage == 'Production':
            for v in self.models.get(name, []):
                if v['stage'] == 'Production' and v['version'] != version:
                    v['stage'] = 'Archived'
                    v['updated_at'] = datetime.now().isoformat()
                    print(f"  Archived: {name} v{v['version']} (was Production)")
        
        model['stage'] = new_stage
        model['updated_at'] = datetime.now().isoformat()
        print(f"  Transitioned: {name} v{version}: {old_stage} → {new_stage}")
    
    def get_production_model(self, name):
        """Get the current production model"""
        for v in self.models.get(name, []):
            if v['stage'] == 'Production':
                return v
        return None
    
    def _get_version(self, name, version):
        """Get a specific model version"""
        for v in self.models.get(name, []):
            if v['version'] == version:
                return v
        return None
    
    def list_models(self, name=None):
        """List all registered models"""
        models_to_show = {name: self.models[name]} if name else self.models
        
        print(f"\n📦 Model Registry")
        print(f"{'Model':<20} {'Version':<10} {'Stage':<15} {'Metrics'}")
        print("-" * 70)
        
        for model_name, versions in models_to_show.items():
            for v in versions:
                metrics_str = str({k: round(val, 4) for k, val in v['metrics'].items()})[:30]
                print(f"{model_name:<20} v{v['version']:<9} {v['stage']:<15} {metrics_str}")


# === Demo: Experiment Tracking ===
print("=" * 60)
print("Part 1: Experiment Tracking")
print("=" * 60)

tracker = ExperimentTracker("sentiment-analysis")

# Simulate multiple training runs
configs = [
    {"name": "baseline", "params": {"model": "bert-base", "lr": 2e-5, "epochs": 3, "batch_size": 32},
     "metrics": {"accuracy": [0.82, 0.87, 0.89], "f1": [0.80, 0.85, 0.88], "loss": [0.45, 0.32, 0.25]}},
    {"name": "larger-lr", "params": {"model": "bert-base", "lr": 5e-5, "epochs": 3, "batch_size": 32},
     "metrics": {"accuracy": [0.85, 0.90, 0.91], "f1": [0.83, 0.89, 0.90], "loss": [0.40, 0.28, 0.22]}},
    {"name": "distilbert", "params": {"model": "distilbert", "lr": 3e-5, "epochs": 5, "batch_size": 64},
     "metrics": {"accuracy": [0.80, 0.84, 0.87, 0.88, 0.88], "f1": [0.78, 0.82, 0.85, 0.86, 0.87], "loss": [0.50, 0.38, 0.30, 0.27, 0.26]}},
]

for config in configs:
    run_id = tracker.start_run(config["name"])
    tracker.log_params(config["params"])
    for metric_name, values in config["metrics"].items():
        for step, value in enumerate(values):
            tracker.log_metric(metric_name, value, step=step)
    tracker.log_artifact(f"models/{config['name']}/model.pt")
    tracker.end_run()

tracker.compare_runs()

best_run, best_acc = tracker.get_best_run("accuracy", mode="max")
print(f"\n🏆 Best run: {best_run['run_name']} (accuracy: {best_acc:.4f})")

# === Part 2: Model Registry ===
print("\n" + "=" * 60)
print("Part 2: Model Registry")
print("=" * 60)

registry = ModelRegistry()

# Register model versions
registry.register_model("sentiment-model", "1.0.0", "abc123",
                        metrics={"accuracy": 0.89, "f1": 0.88, "latency_ms": 45},
                        description="Baseline BERT model")
registry.register_model("sentiment-model", "1.1.0", "def456",
                        metrics={"accuracy": 0.91, "f1": 0.90, "latency_ms": 43},
                        description="Tuned learning rate")
registry.register_model("sentiment-model", "2.0.0", "ghi789",
                        metrics={"accuracy": 0.88, "f1": 0.87, "latency_ms": 22},
                        description="DistilBERT - faster inference")

# Promote through stages
print("\n--- Stage Transitions ---")
registry.transition_stage("sentiment-model", "1.0.0", "Production")
registry.transition_stage("sentiment-model", "1.1.0", "Staging")

# Promote v1.1.0 to production (auto-archives v1.0.0)
print("\n--- Promoting v1.1.0 to Production ---")
registry.transition_stage("sentiment-model", "1.1.0", "Production")

registry.list_models()

prod_model = registry.get_production_model("sentiment-model")
print(f"\n🚀 Current production model: v{prod_model['version']} "
      f"(accuracy: {prod_model['metrics']['accuracy']:.2f})")

print("\n✅ Model versioning and registry complete!")

## 4. A/B测试与金丝雀部署

### 4.1 A/B测试

**目标：** 科学地比较不同模型版本的效果

| 要素 | 说明 |
|------|------|
| **假设** | 新模型比旧模型效果更好 |
| **指标** | 准确率、延迟、用户满意度 |
| **样本量** | 需要足够大以达到统计显著性 |
| **分流** | 随机分配用户到不同组 |

### 4.2 统计显著性

$$z = \frac{\hat{p}_1 - \hat{p}_2}{\sqrt{\hat{p}(1-\hat{p})(\frac{1}{n_1}+\frac{1}{n_2})}}$$

- **p-value < 0.05**: 结果显著，可以采用新模型
- **p-value ≥ 0.05**: 结果不显著，需要更多数据或保持现状

### 4.3 金丝雀部署

```
阶段1: 5% 流量 → 新模型  (观察1小时)
阶段2: 25% 流量 → 新模型 (观察4小时)
阶段3: 50% 流量 → 新模型 (观察12小时)
阶段4: 100% 流量 → 新模型 (全量发布)

任何阶段出现异常 → 自动回滚到旧模型
```

### 4.4 部署策略对比

| 策略 | 风险 | 速度 | 复杂度 | 适用场景 |
|------|------|------|--------|----------|
| **蓝绿部署** | 低 | 快 | 中 | 关键服务 |
| **金丝雀部署** | 很低 | 慢 | 高 | 大规模服务 |
| **滚动更新** | 中 | 中 | 低 | 一般服务 |
| **A/B测试** | 低 | 慢 | 高 | 需要数据驱动决策 |

In [ ]:
# 🔬 Micro Practice: A/B Testing and Canary Deployment
# Goal: Implement traffic splitting and gradual rollout
# Expected outcome: Understand safe model deployment strategies

class ABTestFramework:
    """
    A/B Testing framework for model comparison
    Supports traffic splitting and statistical significance testing
    """
    
    def __init__(self, model_a, model_b, traffic_split=0.5):
        self.model_a = model_a  # Control (current model)
        self.model_b = model_b  # Treatment (new model)
        self.traffic_split = traffic_split  # Fraction going to model B
        self.results_a = []
        self.results_b = []
    
    def route_request(self, x):
        """Route request to model A or B based on traffic split"""
        if np.random.random() < self.traffic_split:
            return 'B', self.model_b(x)
        else:
            return 'A', self.model_a(x)
    
    def record_result(self, model_id, prediction, actual, latency_ms):
        """Record prediction result for analysis"""
        correct = (prediction == actual)
        result = {'correct': correct, 'latency_ms': latency_ms}
        
        if model_id == 'A':
            self.results_a.append(result)
        else:
            self.results_b.append(result)
    
    def compute_statistics(self):
        """Compute comparison statistics"""
        if not self.results_a or not self.results_b:
            return None
        
        acc_a = np.mean([r['correct'] for r in self.results_a])
        acc_b = np.mean([r['correct'] for r in self.results_b])
        lat_a = np.mean([r['latency_ms'] for r in self.results_a])
        lat_b = np.mean([r['latency_ms'] for r in self.results_b])
        
        # Z-test for proportion comparison
        n_a, n_b = len(self.results_a), len(self.results_b)
        p_pooled = (acc_a * n_a + acc_b * n_b) / (n_a + n_b)
        se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_a + 1/n_b))
        z_stat = (acc_b - acc_a) / se if se > 0 else 0
        
        # Approximate p-value (two-tailed)
        p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z_stat) / np.sqrt(2))))
        
        return {
            'model_a': {'accuracy': acc_a, 'latency': lat_a, 'n_samples': n_a},
            'model_b': {'accuracy': acc_b, 'latency': lat_b, 'n_samples': n_b},
            'improvement': acc_b - acc_a,
            'z_statistic': z_stat,
            'p_value': p_value,
            'significant': p_value < 0.05,
        }

class CanaryDeployment:
    """
    Canary (gradual) deployment strategy
    Gradually increase traffic to new model while monitoring metrics
    """
    
    def __init__(self, stages=None):
        self.stages = stages or [
            {'name': 'canary', 'traffic': 0.05, 'duration_hours': 1, 'min_samples': 100},
            {'name': 'early', 'traffic': 0.25, 'duration_hours': 4, 'min_samples': 500},
            {'name': 'half', 'traffic': 0.50, 'duration_hours': 12, 'min_samples': 2000},
            {'name': 'full', 'traffic': 1.00, 'duration_hours': 0, 'min_samples': 0},
        ]
        self.current_stage = 0
        self.stage_metrics = []
    
    def get_current_traffic(self):
        """Get current traffic percentage for new model"""
        if self.current_stage < len(self.stages):
            return self.stages[self.current_stage]['traffic']
        return 1.0
    
    def evaluate_stage(self, accuracy, error_rate, latency_p95):
        """Evaluate if current stage passes health checks"""
        checks = {
            'accuracy_ok': accuracy > 0.80,
            'error_rate_ok': error_rate < 0.01,
            'latency_ok': latency_p95 < 200,
        }
        
        all_passed = all(checks.values())
        
        self.stage_metrics.append({
            'stage': self.stages[self.current_stage]['name'],
            'traffic': self.get_current_traffic(),
            'accuracy': accuracy,
            'error_rate': error_rate,
            'latency_p95': latency_p95,
            'checks': checks,
            'passed': all_passed,
        })
        
        return all_passed, checks
    
    def advance(self):
        """Move to next deployment stage"""
        if self.current_stage < len(self.stages) - 1:
            self.current_stage += 1
            return True
        return False
    
    def rollback(self):
        """Rollback to 0% traffic"""
        self.current_stage = 0
        return "Rolled back to 0% traffic"

# Demo A/B Testing
print("=== A/B Testing ===\n")

np.random.seed(42)

# Simulate two models with different accuracies
def model_a_predict(x):
    return 1 if np.random.random() < 0.82 else 0  # 82% accuracy

def model_b_predict(x):
    return 1 if np.random.random() < 0.86 else 0  # 86% accuracy

ab_test = ABTestFramework(model_a_predict, model_b_predict, traffic_split=0.5)

# Simulate 2000 requests
for i in range(2000):
    x = np.random.randn(10)
    actual = 1 if np.random.random() < 0.85 else 0
    
    model_id, prediction = ab_test.route_request(x)
    latency = np.random.lognormal(3.5, 0.3)
    ab_test.record_result(model_id, prediction, actual, latency)

# Analyze results
stats = ab_test.compute_statistics()

print(f"Model A (Control):  accuracy={stats['model_a']['accuracy']:.3f}, "
      f"n={stats['model_a']['n_samples']}")
print(f"Model B (Treatment): accuracy={stats['model_b']['accuracy']:.3f}, "
      f"n={stats['model_b']['n_samples']}")
print(f"\nImprovement: {stats['improvement']*100:+.1f}%")
print(f"Z-statistic: {stats['z_statistic']:.3f}")
print(f"P-value: {stats['p_value']:.4f}")
print(f"Statistically significant: {'✓ YES' if stats['significant'] else '✗ NO'}")

# Demo Canary Deployment
print("\n=== Canary Deployment ===\n")

canary = CanaryDeployment()

# Simulate deployment stages
print(f"{'Stage':<10} {'Traffic':<10} {'Accuracy':<12} {'Error Rate':<12} {'Latency P95':<12} {'Status':<10}")
print("-" * 66)

for stage in canary.stages:
    traffic = canary.get_current_traffic()
    
    # Simulate metrics (new model is slightly better)
    accuracy = 0.85 + np.random.randn() * 0.02
    error_rate = 0.005 + np.random.random() * 0.003
    latency_p95 = 80 + np.random.randn() * 20
    
    passed, checks = canary.evaluate_stage(accuracy, error_rate, latency_p95)
    status = "✓ PASS" if passed else "✗ FAIL"
    
    print(f"{stage['name']:<10} {traffic*100:<10.0f}% {accuracy:<12.3f} "
          f"{error_rate:<12.4f} {latency_p95:<12.1f}ms {status:<10}")
    
    if passed:
        if not canary.advance():
            print("\n🎉 Full deployment complete!")
            break
    else:
        print(f"\n⚠️ {canary.rollback()}")
        break

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A/B test results
models = ['Model A\n(Control)', 'Model B\n(Treatment)']
accuracies = [stats['model_a']['accuracy'], stats['model_b']['accuracy']]
colors = ['#FF9800', '#4CAF50']

bars = axes[0].bar(models, accuracies, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Accuracy')
axes[0].set_title(f"A/B Test Results (p={stats['p_value']:.4f})", fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, acc + 0.005, 
                f'{acc:.3f}', ha='center', fontsize=12, weight='bold')

sig_text = "Significant ✓" if stats['significant'] else "Not Significant ✗"
axes[0].text(0.5, 0.95, sig_text, transform=axes[0].transAxes, ha='center',
            fontsize=12, color='green' if stats['significant'] else 'red',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Canary deployment stages
stage_names = [m['stage'] for m in canary.stage_metrics]
stage_traffic = [m['traffic'] * 100 for m in canary.stage_metrics]
stage_acc = [m['accuracy'] for m in canary.stage_metrics]

ax2 = axes[1].twinx()
axes[1].bar(stage_names, stage_traffic, color='steelblue', alpha=0.5, label='Traffic %')
ax2.plot(stage_names, stage_acc, 'ro-', linewidth=2, markersize=10, label='Accuracy')
ax2.axhline(y=0.80, color='red', linestyle='--', alpha=0.5, label='Min Accuracy')

axes[1].set_ylabel('Traffic (%)', color='steelblue')
ax2.set_ylabel('Accuracy', color='red')
axes[1].set_title('Canary Deployment Progress', fontsize=12, weight='bold')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nA/B测试与灰度发布要点：")
print("1. A/B测试需要足够样本量确保统计显著性")
print("2. 灰度发布从小流量开始，逐步扩大")
print("3. 每个阶段都要检查健康指标")
print("4. 发现问题立即回滚，不要犹豫")
print("5. 记录所有实验结果，积累经验")
print("\n✓ A/B测试与灰度发布完成！")

## 5. 模型监控与漂移检测

### 5.1 监控维度

| 维度 | 指标 | 告警条件 |
|------|------|---------|
| **系统性能** | 延迟、吞吐量、CPU/GPU利用率 | P95延迟超阈值 |
| **模型质量** | 准确率、F1、AUC | 指标下降>5% |
| **数据质量** | 缺失值、异常值、分布变化 | 漂移检测触发 |
| **业务指标** | 转化率、用户满意度 | 业务KPI下降 |

### 5.2 数据漂移类型

**数据漂移（Data Drift）**：
- 输入数据分布变化：$P_{train}(X) \neq P_{prod}(X)$
- 原因：用户行为变化、数据源变更
- 检测：KS检验、PSI

**概念漂移（Concept Drift）**：
- 输入输出关系变化：$P_{train}(Y|X) \neq P_{prod}(Y|X)$
- 原因：业务规则变化、外部环境变化
- 检测：监控模型性能指标

### 5.3 漂移检测方法

**KS检验（Kolmogorov-Smirnov Test）**：
- 比较两个分布的最大差异
- $D = \max|F_{ref}(x) - F_{cur}(x)|$
- D > 0.15 通常表示显著漂移

**PSI（Population Stability Index）**：
- $PSI = \sum (P_i - Q_i) \ln\frac{P_i}{Q_i}$
- PSI < 0.1：无漂移
- PSI 0.1-0.25：中等漂移
- PSI > 0.25：显著漂移

### 5.4 告警策略

```
INFO:     PSI < 0.1, 性能正常
WARNING:  PSI 0.1-0.25, 或性能下降2-5%
CRITICAL: PSI > 0.25, 或性能下降>5%
ACTION:   自动触发重训练评估
```

In [ ]:
# 🔬 Micro Practice: Model Monitoring and Drift Detection
# Goal: Implement drift detection and alerting
# Expected outcome: Understand how to detect model degradation

class DriftDetector:
    """
    Detect data drift using statistical tests
    Monitors input distribution changes that may affect model performance
    """
    
    @staticmethod
    def ks_test(reference, current):
        """
        Kolmogorov-Smirnov test for distribution comparison
        Returns: statistic (0-1, higher = more drift), p-value
        """
        # Sort both samples
        ref_sorted = np.sort(reference)
        cur_sorted = np.sort(current)
        
        # Compute empirical CDFs
        n_ref = len(ref_sorted)
        n_cur = len(cur_sorted)
        
        # Combine and sort all values
        all_values = np.sort(np.concatenate([ref_sorted, cur_sorted]))
        
        # Compute CDFs at each point
        cdf_ref = np.searchsorted(ref_sorted, all_values, side='right') / n_ref
        cdf_cur = np.searchsorted(cur_sorted, all_values, side='right') / n_cur
        
        # KS statistic = max difference between CDFs
        ks_stat = np.max(np.abs(cdf_ref - cdf_cur))
        
        # Approximate p-value
        n_eff = np.sqrt(n_ref * n_cur / (n_ref + n_cur))
        p_value = np.exp(-2 * (n_eff * ks_stat) ** 2)
        
        return ks_stat, p_value
    
    @staticmethod
    def psi(reference, current, n_bins=10):
        """
        Population Stability Index
        PSI < 0.1: no drift
        PSI 0.1-0.25: moderate drift
        PSI > 0.25: significant drift
        """
        # Create bins from reference distribution
        bins = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
        bins[0] = -np.inf
        bins[-1] = np.inf
        
        # Count proportions in each bin
        ref_counts = np.histogram(reference, bins=bins)[0] / len(reference)
        cur_counts = np.histogram(current, bins=bins)[0] / len(current)
        
        # Avoid division by zero
        ref_counts = np.clip(ref_counts, 1e-6, None)
        cur_counts = np.clip(cur_counts, 1e-6, None)
        
        # PSI formula
        psi_value = np.sum((cur_counts - ref_counts) * np.log(cur_counts / ref_counts))
        
        return psi_value

class ModelMonitor:
    """
    Comprehensive model monitoring system
    Tracks performance, data drift, and system health
    """
    
    def __init__(self, reference_data, alert_thresholds=None):
        self.reference_data = reference_data
        self.drift_detector = DriftDetector()
        self.metrics_history = []
        self.alerts = []
        
        self.thresholds = alert_thresholds or {
            'ks_stat': 0.15,
            'psi': 0.25,
            'accuracy_drop': 0.05,
            'latency_p95_ms': 200,
        }
    
    def check_drift(self, current_data):
        """Check for data drift across all features"""
        results = {}
        n_features = min(current_data.shape[1], self.reference_data.shape[1])
        
        for i in range(n_features):
            ref_feat = self.reference_data[:, i]
            cur_feat = current_data[:, i]
            
            ks_stat, p_value = self.drift_detector.ks_test(ref_feat, cur_feat)
            psi_value = self.drift_detector.psi(ref_feat, cur_feat)
            
            results[f'feature_{i}'] = {
                'ks_statistic': ks_stat,
                'ks_p_value': p_value,
                'psi': psi_value,
                'drift_detected': ks_stat > self.thresholds['ks_stat'] or psi_value > self.thresholds['psi'],
            }
        
        return results
    
    def record_metrics(self, accuracy, latency_ms, predictions, labels):
        """Record monitoring metrics"""
        self.metrics_history.append({
            'timestamp': datetime.now().isoformat(),
            'accuracy': accuracy,
            'latency_ms': latency_ms,
            'n_predictions': len(predictions),
        })
    
    def generate_alerts(self, drift_results, current_accuracy=None):
        """Generate alerts based on monitoring results"""
        new_alerts = []
        
        # Check drift alerts
        drifted_features = [k for k, v in drift_results.items() if v['drift_detected']]
        if len(drifted_features) > 0:
            severity = 'CRITICAL' if len(drifted_features) > 3 else 'WARNING'
            new_alerts.append({
                'type': 'DATA_DRIFT',
                'severity': severity,
                'message': f"Drift detected in {len(drifted_features)} features: {drifted_features[:3]}",
                'timestamp': datetime.now().isoformat(),
            })
        
        # Check accuracy
        if current_accuracy is not None and self.metrics_history:
            baseline_acc = self.metrics_history[0].get('accuracy', 1.0)
            if baseline_acc - current_accuracy > self.thresholds['accuracy_drop']:
                new_alerts.append({
                    'type': 'ACCURACY_DROP',
                    'severity': 'CRITICAL',
                    'message': f"Accuracy dropped: {baseline_acc:.3f} → {current_accuracy:.3f}",
                    'timestamp': datetime.now().isoformat(),
                })
        
        self.alerts.extend(new_alerts)
        return new_alerts

# Demo monitoring and drift detection
print("=== Model Monitoring & Drift Detection ===\n")

np.random.seed(42)

# Reference data (training distribution)
n_samples = 1000
n_features = 5
reference_data = np.random.randn(n_samples, n_features)

# Scenario 1: No drift
print("--- Scenario 1: No Drift ---")
current_no_drift = np.random.randn(500, n_features)

monitor = ModelMonitor(reference_data)
results_no_drift = monitor.check_drift(current_no_drift)

for feat, result in list(results_no_drift.items())[:3]:
    print(f"  {feat}: KS={result['ks_statistic']:.3f}, PSI={result['psi']:.3f}, "
          f"Drift={'✗ YES' if result['drift_detected'] else '✓ No'}")

# Scenario 2: Gradual drift (mean shift)
print("\n--- Scenario 2: Gradual Drift (mean shift) ---")
current_drift = np.random.randn(500, n_features) + 0.5  # Shifted mean

results_drift = monitor.check_drift(current_drift)

for feat, result in list(results_drift.items())[:3]:
    print(f"  {feat}: KS={result['ks_statistic']:.3f}, PSI={result['psi']:.3f}, "
          f"Drift={'✗ YES' if result['drift_detected'] else '✓ No'}")

# Scenario 3: Sudden drift (distribution change)
print("\n--- Scenario 3: Sudden Drift (distribution change) ---")
current_sudden = np.random.exponential(1.0, (500, n_features))  # Different distribution

results_sudden = monitor.check_drift(current_sudden)

for feat, result in list(results_sudden.items())[:3]:
    print(f"  {feat}: KS={result['ks_statistic']:.3f}, PSI={result['psi']:.3f}, "
          f"Drift={'✗ YES' if result['drift_detected'] else '✓ No'}")

# Generate alerts
print("\n--- Alerts ---")
alerts = monitor.generate_alerts(results_sudden, current_accuracy=0.72)
monitor.record_metrics(accuracy=0.72, latency_ms=45, predictions=[], labels=[])

for alert in alerts:
    print(f"  [{alert['severity']}] {alert['type']}: {alert['message']}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feature distribution comparison
feat_idx = 0
axes[0].hist(reference_data[:, feat_idx], bins=50, alpha=0.5, label='Reference', color='blue', density=True)
axes[0].hist(current_no_drift[:, feat_idx], bins=50, alpha=0.5, label='No Drift', color='green', density=True)
axes[0].hist(current_drift[:, feat_idx], bins=50, alpha=0.5, label='Gradual Drift', color='orange', density=True)
axes[0].hist(current_sudden[:, feat_idx], bins=50, alpha=0.5, label='Sudden Drift', color='red', density=True)
axes[0].set_title('Feature 0 Distribution', fontsize=12, weight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# KS statistics across scenarios
scenarios = ['No Drift', 'Gradual', 'Sudden']
all_results = [results_no_drift, results_drift, results_sudden]
ks_means = [np.mean([v['ks_statistic'] for v in r.values()]) for r in all_results]
psi_means = [np.mean([v['psi'] for v in r.values()]) for r in all_results]

x = np.arange(len(scenarios))
width = 0.35

axes[1].bar(x - width/2, ks_means, width, label='KS Statistic', color='steelblue', alpha=0.7)
axes[1].bar(x + width/2, psi_means, width, label='PSI', color='coral', alpha=0.7)
axes[1].axhline(y=0.15, color='orange', linestyle='--', label='KS Threshold')
axes[1].axhline(y=0.25, color='red', linestyle='--', label='PSI Threshold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenarios)
axes[1].set_title('Drift Metrics by Scenario', fontsize=12, weight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis='y')

# Simulated accuracy over time with drift
days = np.arange(60)
accuracy_timeline = np.ones(60) * 0.85
# Gradual drift starts at day 20
for i in range(20, 60):
    accuracy_timeline[i] = 0.85 - (i - 20) * 0.003 + np.random.randn() * 0.005

axes[2].plot(days, accuracy_timeline, 'b-', linewidth=2)
axes[2].axhline(y=0.85, color='green', linestyle='--', alpha=0.5, label='Baseline')
axes[2].axhline(y=0.80, color='red', linestyle='--', alpha=0.5, label='Alert Threshold')
axes[2].axvspan(20, 60, alpha=0.1, color='red', label='Drift Period')
axes[2].set_xlabel('Days')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Model Accuracy Over Time', fontsize=12, weight='bold')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n监控最佳实践：")
print("1. 同时监控数据漂移和模型性能")
print("2. KS检验适合连续特征，PSI适合分箱特征")
print("3. 设置分级告警：WARNING → CRITICAL")
print("4. 漂移检测应自动触发重训练评估")
print("5. 保存参考数据集用于持续对比")
print("\n✓ 监控与漂移检测完成！")

## 6. 错误处理与调试

### 6.1 错误类型

| 类型 | 来源 | 示例 | 处理策略 |
|------|------|------|---------|
| **输入错误** | 用户/上游 | 格式错误、缺失字段、异常值 | 验证+拒绝 |
| **模型错误** | 模型推理 | NaN输出、超时、OOM | 降级+回退 |
| **系统错误** | 基础设施 | 服务宕机、网络超时 | 重试+熔断 |

### 6.2 防御性编程

**输入验证**：
```python
# 检查类型、维度、范围、NaN/Inf
if not isinstance(x, torch.Tensor):
    raise ValueError(f"Expected Tensor, got {type(x)}")
if torch.isnan(x).any():
    raise ValueError("Input contains NaN")
```

**优雅降级**：
```python
try:
    result = model.predict(x)
except ModelError:
    result = fallback_model.predict(x)  # 备用模型
except Exception:
    result = cached_result  # 缓存结果
```

### 6.3 调试工具

| 工具 | 用途 | 关键功能 |
|------|------|---------|
| **结构化日志** | 记录请求详情 | request_id, 输入摘要, 输出, 延迟 |
| **请求追踪** | 跟踪请求链路 | 分布式追踪, span, trace_id |
| **错误聚合** | 分析错误模式 | 错误分类, 频率, 影响范围 |
| **离线复现** | 重现生产问题 | 保存异常输入, 模型快照 |

### 6.4 熔断器模式

```
正常 → 错误率超阈值 → 熔断（拒绝请求）
                         ↓ 等待冷却期
                      半开（允许少量请求）
                         ↓ 成功率恢复
                      正常（恢复服务）
```

In [ ]:
# 🔬 Micro Practice: Error Handling and Debugging
# Goal: Implement robust error handling for production ML systems
# Expected outcome: Understand graceful degradation and debugging

import traceback
import logging

class InputValidator:
    """Validate model inputs before inference"""
    
    def __init__(self, expected_dim, min_val=-100, max_val=100):
        self.expected_dim = expected_dim
        self.min_val = min_val
        self.max_val = max_val
    
    def validate(self, x):
        """Validate input tensor"""
        errors = []
        
        # Check type
        if not isinstance(x, (torch.Tensor, np.ndarray)):
            errors.append(f"Invalid type: {type(x)}, expected Tensor or ndarray")
            return False, errors
        
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x)
        
        # Check dimensions
        if x.dim() < 1:
            errors.append(f"Invalid dimensions: {x.dim()}, expected >= 1")
        
        if x.shape[-1] != self.expected_dim:
            errors.append(f"Invalid feature dim: {x.shape[-1]}, expected {self.expected_dim}")
        
        # Check values
        if torch.isnan(x).any():
            errors.append(f"Input contains NaN values ({torch.isnan(x).sum().item()} NaNs)")
        
        if torch.isinf(x).any():
            errors.append(f"Input contains Inf values")
        
        if x.min() < self.min_val or x.max() > self.max_val:
            errors.append(f"Values out of range [{self.min_val}, {self.max_val}]")
        
        return len(errors) == 0, errors

class RobustModelServer:
    """
    Production model server with comprehensive error handling
    Implements graceful degradation and fallback strategies
    """
    
    def __init__(self, model, input_dim, fallback_response=None):
        self.model = model
        self.model.eval()
        self.validator = InputValidator(input_dim)
        self.fallback_response = fallback_response
        self.request_log = []
        self.error_counts = {'input_error': 0, 'model_error': 0, 'system_error': 0}
        self.cache = {}
    
    def predict(self, x, request_id=None):
        """Make prediction with full error handling"""
        request_id = request_id or f"req-{len(self.request_log)}"
        log_entry = {
            'request_id': request_id,
            'status': 'pending',
            'error': None,
            'fallback_used': False,
        }
        
        try:
            # Step 1: Input validation
            is_valid, errors = self.validator.validate(x)
            if not is_valid:
                self.error_counts['input_error'] += 1
                log_entry['status'] = 'input_error'
                log_entry['error'] = '; '.join(errors)
                self.request_log.append(log_entry)
                return self._fallback(request_id, f"Input validation failed: {errors}")
            
            if isinstance(x, np.ndarray):
                x = torch.from_numpy(x).float()
            
            # Step 2: Check cache
            cache_key = hash(x.numpy().tobytes())
            if cache_key in self.cache:
                log_entry['status'] = 'cache_hit'
                self.request_log.append(log_entry)
                return self.cache[cache_key]
            
            # Step 3: Model inference
            with torch.no_grad():
                output = self.model(x)
            
            # Step 4: Output validation
            if torch.isnan(output).any():
                self.error_counts['model_error'] += 1
                log_entry['status'] = 'model_error'
                log_entry['error'] = 'Model produced NaN output'
                self.request_log.append(log_entry)
                return self._fallback(request_id, "Model produced NaN")
            
            # Success
            result = output.numpy()
            self.cache[cache_key] = result  # Cache result
            log_entry['status'] = 'success'
            self.request_log.append(log_entry)
            return result
            
        except RuntimeError as e:
            # OOM, CUDA errors, etc.
            self.error_counts['system_error'] += 1
            log_entry['status'] = 'system_error'
            log_entry['error'] = str(e)
            self.request_log.append(log_entry)
            return self._fallback(request_id, f"System error: {e}")
        
        except Exception as e:
            # Unexpected errors
            self.error_counts['system_error'] += 1
            log_entry['status'] = 'unknown_error'
            log_entry['error'] = traceback.format_exc()
            self.request_log.append(log_entry)
            return self._fallback(request_id, f"Unexpected error: {e}")
    
    def _fallback(self, request_id, reason):
        """Return fallback response when prediction fails"""
        if self.fallback_response is not None:
            return self.fallback_response
        return None
    
    def get_error_report(self):
        """Generate error analysis report"""
        total = len(self.request_log)
        if total == 0:
            return "No requests recorded"
        
        status_counts = {}
        for entry in self.request_log:
            status = entry['status']
            status_counts[status] = status_counts.get(status, 0) + 1
        
        report = f"Total requests: {total}\n"
        for status, count in sorted(status_counts.items()):
            report += f"  {status}: {count} ({count/total*100:.1f}%)\n"
        report += f"\nError breakdown: {self.error_counts}"
        return report

# Demo error handling
print("=== Robust Error Handling ===\n")

# Create model and server
model = torch.nn.Sequential(
    torch.nn.Linear(64, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 10)
)

server = RobustModelServer(
    model, input_dim=64,
    fallback_response=np.zeros(10)  # Default fallback
)

# Test various scenarios
print("Testing different input scenarios:\n")

# 1. Valid input
result = server.predict(torch.randn(1, 64), "req-001")
print(f"✓ Valid input: shape={result.shape}")

# 2. Wrong dimensions
result = server.predict(torch.randn(1, 32), "req-002")
print(f"✗ Wrong dim: fallback={'used' if result is not None else 'none'}")

# 3. NaN input
nan_input = torch.randn(1, 64)
nan_input[0, 0] = float('nan')
result = server.predict(nan_input, "req-003")
print(f"✗ NaN input: fallback={'used' if result is not None else 'none'}")

# 4. String input (wrong type)
result = server.predict("invalid", "req-004")
print(f"✗ Wrong type: fallback={'used' if result is not None else 'none'}")

# 5. Numpy array input
result = server.predict(np.random.randn(1, 64).astype(np.float32), "req-005")
print(f"✓ Numpy input: shape={result.shape}")

# 6. Cache hit
result = server.predict(torch.randn(1, 64), "req-006")
result2 = server.predict(torch.randn(1, 64), "req-007")

# Simulate many requests
print("\nSimulating 1000 requests with mixed quality...")
np.random.seed(42)
for i in range(1000):
    r = np.random.random()
    if r < 0.9:  # 90% valid
        server.predict(torch.randn(1, 64), f"batch-{i}")
    elif r < 0.95:  # 5% wrong dim
        server.predict(torch.randn(1, 32), f"batch-{i}")
    elif r < 0.98:  # 3% NaN
        bad = torch.randn(1, 64)
        bad[0, 0] = float('nan')
        server.predict(bad, f"batch-{i}")
    else:  # 2% wrong type
        server.predict("bad", f"batch-{i}")

# Error report
print("\n" + server.get_error_report())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error distribution
status_counts = {}
for entry in server.request_log:
    s = entry['status']
    status_counts[s] = status_counts.get(s, 0) + 1

statuses = list(status_counts.keys())
counts = list(status_counts.values())
colors = ['#4CAF50' if s == 'success' or s == 'cache_hit' else '#F44336' for s in statuses]

axes[0].barh(statuses, counts, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Count')
axes[0].set_title('Request Status Distribution', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Error types over time (rolling window)
window = 50
error_rate = []
for i in range(window, len(server.request_log)):
    window_entries = server.request_log[i-window:i]
    errors = sum(1 for e in window_entries if e['status'] not in ('success', 'cache_hit'))
    error_rate.append(errors / window * 100)

axes[1].plot(error_rate, 'r-', linewidth=1, alpha=0.7)
axes[1].axhline(y=5, color='orange', linestyle='--', label='Warning (5%)')
axes[1].axhline(y=10, color='red', linestyle='--', label='Critical (10%)')
axes[1].set_xlabel('Request Number')
axes[1].set_ylabel('Error Rate (%)')
axes[1].set_title(f'Rolling Error Rate (window={window})', fontsize=12, weight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("错误处理最佳实践：")
print("1. 输入验证：在推理前检查所有输入")
print("2. 优雅降级：失败时返回合理的默认值")
print("3. 结构化日志：记录每个请求的完整信息")
print("4. 错误分类：区分输入错误、模型错误、系统错误")
print("5. 监控错误率：设置告警阈值")
print("\n✓ 错误处理与调试完成！")

## 7. SLA与性能优化

### 7.1 SLA/SLO/SLI

| 概念 | 定义 | 示例 |
|------|------|------|
| **SLA** (Service Level Agreement) | 与客户的服务承诺 | "99.9%可用性" |
| **SLO** (Service Level Objective) | 内部服务目标 | "P95延迟<100ms" |
| **SLI** (Service Level Indicator) | 实际测量指标 | "当前P95=85ms" |

### 7.2 关键SLO指标

**可用性**：
- 99.9% = 每月约43分钟停机
- 99.99% = 每月约4.3分钟停机
- 99.999% = 每月约26秒停机

**延迟**：
- P50：中位数延迟（典型体验）
- P95：95%请求的延迟上限
- P99：长尾延迟（最差体验）

**Error Budget**：
- 允许的错误预算 = 1 - SLO
- 例：99.9% SLO → 0.1% error budget
- 用完预算 → 冻结新功能，专注稳定性

### 7.3 性能优化策略

| 策略 | 效果 | 适用场景 |
|------|------|---------|
| **响应缓存** | 延迟降低10x | 重复请求多 |
| **请求批处理** | 吞吐量提升3-5x | 高并发场景 |
| **自动扩缩容** | 应对流量波动 | 流量不均匀 |
| **模型优化** | 延迟降低2-4x | 模型是瓶颈 |
| **CDN/边缘部署** | 延迟降低50% | 地理分布用户 |

In [ ]:
# 🔬 Micro Practice: SLA Monitoring and Performance Optimization
# Goal: Implement SLA tracking and optimization strategies
# Expected outcome: Understand how to maintain production SLAs

class SLAMonitor:
    """
    Service Level Agreement monitoring
    Track SLIs (Service Level Indicators) against SLOs (Service Level Objectives)
    """
    
    def __init__(self, sla_config):
        self.config = sla_config
        self.metrics_history = []
    
    def record_request(self, latency_ms, success, timestamp=None):
        """Record a single request's metrics"""
        self.metrics_history.append({
            'timestamp': timestamp or datetime.now().isoformat(),
            'latency_ms': latency_ms,
            'success': success,
        })
    
    def calculate_sli(self, window_minutes=60):
        """Calculate current SLIs"""
        if not self.metrics_history:
            return {}
        
        recent = self.metrics_history  # In production, filter by time window
        latencies = [m['latency_ms'] for m in recent]
        successes = [m['success'] for m in recent]
        
        return {
            'availability': sum(successes) / len(successes) * 100,
            'latency_p50': np.percentile(latencies, 50),
            'latency_p95': np.percentile(latencies, 95),
            'latency_p99': np.percentile(latencies, 99),
            'throughput': len(recent),  # requests in window
            'error_rate': (1 - sum(successes) / len(successes)) * 100,
        }
    
    def check_slo_compliance(self):
        """Check if current SLIs meet SLOs"""
        sli = self.calculate_sli()
        if not sli:
            return {}
        
        violations = []
        
        # Check availability
        if sli['availability'] < self.config['availability_target']:
            violations.append({
                'metric': 'Availability',
                'target': f"{self.config['availability_target']}%",
                'actual': f"{sli['availability']:.2f}%",
                'severity': 'CRITICAL' if sli['availability'] < 99.0 else 'WARNING',
            })
        
        # Check latency
        if sli['latency_p95'] > self.config['latency_p95_ms']:
            violations.append({
                'metric': 'Latency P95',
                'target': f"{self.config['latency_p95_ms']}ms",
                'actual': f"{sli['latency_p95']:.1f}ms",
                'severity': 'WARNING',
            })
        
        if sli['latency_p99'] > self.config['latency_p99_ms']:
            violations.append({
                'metric': 'Latency P99',
                'target': f"{self.config['latency_p99_ms']}ms",
                'actual': f"{sli['latency_p99']:.1f}ms",
                'severity': 'CRITICAL',
            })
        
        # Check error rate
        if sli['error_rate'] > self.config['max_error_rate']:
            violations.append({
                'metric': 'Error Rate',
                'target': f"<{self.config['max_error_rate']}%",
                'actual': f"{sli['error_rate']:.2f}%",
                'severity': 'CRITICAL',
            })
        
        return {
            'sli': sli,
            'compliant': len(violations) == 0,
            'violations': violations,
        }

class PerformanceOptimizer:
    """Strategies for maintaining SLA compliance"""
    
    @staticmethod
    def simulate_caching(latencies, cache_hit_rate=0.7):
        """Simulate effect of response caching"""
        cached_latencies = []
        for lat in latencies:
            if np.random.random() < cache_hit_rate:
                cached_latencies.append(lat * 0.1)  # Cache hit: 10x faster
            else:
                cached_latencies.append(lat)
        return cached_latencies
    
    @staticmethod
    def simulate_batching(latencies, batch_size=4):
        """Simulate effect of request batching"""
        # Batching increases individual latency but improves throughput
        batched = []
        for i in range(0, len(latencies), batch_size):
            batch = latencies[i:i+batch_size]
            batch_latency = max(batch) * 1.2  # Slight overhead
            batched.extend([batch_latency] * len(batch))
        return batched
    
    @staticmethod
    def simulate_autoscaling(latencies, threshold_ms=100):
        """Simulate effect of auto-scaling"""
        scaled = []
        scale_factor = 1.0
        for lat in latencies:
            if lat > threshold_ms:
                scale_factor = min(scale_factor + 0.1, 3.0)  # Scale up
            else:
                scale_factor = max(scale_factor - 0.05, 1.0)  # Scale down
            scaled.append(lat / scale_factor)
        return scaled

# Demo SLA monitoring
print("=== SLA Monitoring ===\n")

sla_config = {
    'availability_target': 99.9,    # 99.9% uptime
    'latency_p95_ms': 100,          # P95 < 100ms
    'latency_p99_ms': 200,          # P99 < 200ms
    'max_error_rate': 0.1,          # < 0.1% errors
    'min_throughput': 1000,          # > 1000 req/s
}

print("SLA Configuration:")
for key, value in sla_config.items():
    print(f"  {key}: {value}")

monitor = SLAMonitor(sla_config)

# Simulate normal traffic
print("\n--- Normal Traffic ---")
np.random.seed(42)
for _ in range(1000):
    latency = np.random.lognormal(mean=3.5, sigma=0.5)  # ~30-50ms typical
    success = np.random.random() > 0.0005  # 99.95% success rate
    monitor.record_request(latency, success)

result = monitor.check_slo_compliance()
sli = result['sli']

print(f"  Availability: {sli['availability']:.2f}% (target: {sla_config['availability_target']}%)")
print(f"  Latency P50:  {sli['latency_p50']:.1f}ms")
print(f"  Latency P95:  {sli['latency_p95']:.1f}ms (target: <{sla_config['latency_p95_ms']}ms)")
print(f"  Latency P99:  {sli['latency_p99']:.1f}ms (target: <{sla_config['latency_p99_ms']}ms)")
print(f"  Error Rate:   {sli['error_rate']:.3f}%")
print(f"  SLO Compliant: {'✓' if result['compliant'] else '✗'}")

if result['violations']:
    print(f"\n  Violations:")
    for v in result['violations']:
        print(f"    [{v['severity']}] {v['metric']}: {v['actual']} (target: {v['target']})")

# Simulate degraded traffic
print("\n--- Degraded Traffic (with optimization) ---")
degraded_latencies = np.random.lognormal(mean=4.5, sigma=0.8, size=1000)  # Higher latency

optimizer = PerformanceOptimizer()
cached = optimizer.simulate_caching(degraded_latencies, cache_hit_rate=0.6)
scaled = optimizer.simulate_autoscaling(degraded_latencies, threshold_ms=100)

print(f"  Original P95:    {np.percentile(degraded_latencies, 95):.1f}ms")
print(f"  With Caching:    {np.percentile(cached, 95):.1f}ms")
print(f"  With Autoscale:  {np.percentile(scaled, 95):.1f}ms")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# SLI Dashboard
metrics = ['Availability\n(%)', 'Latency P95\n(ms)', 'Latency P99\n(ms)', 'Error Rate\n(%)']
actuals = [sli['availability'], sli['latency_p95'], sli['latency_p99'], sli['error_rate']]
targets = [sla_config['availability_target'], sla_config['latency_p95_ms'], 
           sla_config['latency_p99_ms'], sla_config['max_error_rate']]

x = np.arange(len(metrics))
width = 0.35

bars1 = axes[0].bar(x - width/2, actuals, width, label='Actual', color='steelblue', alpha=0.7)
bars2 = axes[0].bar(x + width/2, targets, width, label='Target', color='coral', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, fontsize=9)
axes[0].set_title('SLI vs SLO Dashboard', fontsize=12, weight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Latency distribution comparison
axes[1].hist(degraded_latencies, bins=50, alpha=0.5, label='Original', color='red', density=True)
axes[1].hist(cached, bins=50, alpha=0.5, label='With Caching', color='green', density=True)
axes[1].axvline(x=sla_config['latency_p95_ms'], color='black', linestyle='--', label='P95 Target')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_ylabel('Density')
axes[1].set_title('Optimization Effect on Latency', fontsize=12, weight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 500)

# Optimization comparison
strategies = ['Original', 'Caching\n(60%)', 'Auto-\nscaling']
p95s = [np.percentile(degraded_latencies, 95), np.percentile(cached, 95), np.percentile(scaled, 95)]
colors = ['#F44336', '#4CAF50', '#2196F3']

axes[2].bar(strategies, p95s, color=colors, alpha=0.7, edgecolor='black')
axes[2].axhline(y=sla_config['latency_p95_ms'], color='red', linestyle='--', label='P95 Target')
axes[2].set_ylabel('P95 Latency (ms)')
axes[2].set_title('Optimization Strategies', fontsize=12, weight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nSLA管理最佳实践：")
print("1. 定义清晰的SLO（不要过于激进）")
print("2. 持续监控SLI，设置预警阈值")
print("3. 使用Error Budget管理风险")
print("4. 缓存是最简单有效的优化手段")
print("5. 自动扩缩容应对流量波动")
print("\n✓ SLA监控与优化完成！")

## 8. 事故响应与复盘

### 8.1 事故管理流程

```
检测 (Detection)
  ↓ 监控告警触发
分级 (Triage)
  ↓ 评估影响范围和严重程度
缓解 (Mitigation)
  ↓ 快速恢复服务（回滚、降级）
调查 (Investigation)
  ↓ 找到根本原因
恢复 (Resolution)
  ↓ 彻底修复问题
复盘 (Postmortem)
  → 总结经验，制定改进措施
```

### 8.2 事故严重程度

| 级别 | 定义 | 响应时间 | 示例 |
|------|------|---------|------|
| **P0** | 服务完全不可用 | <15分钟 | 模型服务宕机 |
| **P1** | 核心功能受损 | <30分钟 | 模型准确率大幅下降 |
| **P2** | 次要功能受损 | <2小时 | 部分用户延迟增加 |
| **P3** | 轻微问题 | <24小时 | 日志格式异常 |

### 8.3 Runbook（运维手册）

**预定义的事故处理步骤**，确保：
- 新人也能快速响应
- 减少人为判断错误
- 标准化处理流程
- 缩短恢复时间

### 8.4 复盘（Postmortem）

**复盘模板**：
1. **概述**：什么时候发生了什么
2. **影响**：影响了多少用户/多长时间
3. **时间线**：详细的事件经过
4. **根本原因**：为什么会发生
5. **缓解措施**：如何恢复的
6. **Action Items**：如何防止再次发生

**核心原则**：
- 无责复盘（Blameless Postmortem）
- 关注系统改进，而非个人失误
- 每个Action Item必须有Owner和截止日期

In [ ]:
# 🎯 Comprehensive Example: Incident Response System
# Goal: Build incident management and postmortem framework
# Expected outcome: Understand production incident handling

class IncidentSeverity:
    """Incident severity levels"""
    P0_CRITICAL = 0  # Service down, all users affected
    P1_HIGH = 1      # Major feature broken, many users affected
    P2_MEDIUM = 2    # Minor feature broken, some users affected
    P3_LOW = 3       # Cosmetic issue, few users affected

class Incident:
    """Represent a production incident"""
    def __init__(self, title, severity, description):
        self.id = f"INC-{np.random.randint(1000, 9999)}"
        self.title = title
        self.severity = severity
        self.description = description
        self.status = "detected"
        self.timeline = [(datetime.now().isoformat(), "detected", description)]
        self.mitigation = None
        self.root_cause = None
        self.action_items = []
    
    def update_status(self, status, note=""):
        self.status = status
        self.timeline.append((datetime.now().isoformat(), status, note))
    
    def set_mitigation(self, mitigation):
        self.mitigation = mitigation
        self.update_status("mitigated", mitigation)
    
    def set_root_cause(self, root_cause):
        self.root_cause = root_cause
        self.update_status("root_cause_identified", root_cause)
    
    def resolve(self, resolution):
        self.update_status("resolved", resolution)
    
    def add_action_item(self, item, owner, priority):
        self.action_items.append({
            'item': item, 'owner': owner, 'priority': priority, 'status': 'open'
        })

class IncidentResponseSystem:
    """Production incident response framework"""
    
    def __init__(self):
        self.incidents = []
        self.runbooks = self._create_runbooks()
    
    def _create_runbooks(self):
        """Pre-defined runbooks for common incidents"""
        return {
            'model_degradation': {
                'title': 'Model Performance Degradation',
                'steps': [
                    '1. Check monitoring dashboard for metric drops',
                    '2. Compare current vs baseline metrics',
                    '3. Check for data drift (run drift detection)',
                    '4. Check recent model/data changes',
                    '5. If drift detected: trigger retraining pipeline',
                    '6. If no drift: check for system issues',
                    '7. Consider rollback to previous model version',
                ],
                'rollback': 'Switch to previous model version via model registry',
            },
            'service_outage': {
                'title': 'Model Service Outage',
                'steps': [
                    '1. Check service health endpoints',
                    '2. Check infrastructure (CPU, memory, disk)',
                    '3. Check dependency services (database, cache)',
                    '4. Review recent deployments',
                    '5. Restart service if needed',
                    '6. Scale up if load-related',
                    '7. Enable fallback/cached responses',
                ],
                'rollback': 'Revert to last known good deployment',
            },
            'data_pipeline_failure': {
                'title': 'Data Pipeline Failure',
                'steps': [
                    '1. Check pipeline logs for errors',
                    '2. Verify data source availability',
                    '3. Check data format/schema changes',
                    '4. Validate data quality metrics',
                    '5. Rerun failed pipeline steps',
                    '6. Use cached/backup data if available',
                    '7. Alert data team for investigation',
                ],
                'rollback': 'Use last successful data snapshot',
            },
            'high_latency': {
                'title': 'High Latency / SLA Violation',
                'steps': [
                    '1. Check current latency percentiles (P50, P95, P99)',
                    '2. Check request volume (traffic spike?)',
                    '3. Check model inference time',
                    '4. Check downstream dependencies',
                    '5. Enable request throttling if needed',
                    '6. Scale up compute resources',
                    '7. Enable response caching',
                ],
                'rollback': 'Switch to lighter model version',
            },
        }
    
    def create_incident(self, title, severity, description):
        incident = Incident(title, severity, description)
        self.incidents.append(incident)
        return incident
    
    def get_runbook(self, incident_type):
        return self.runbooks.get(incident_type)
    
    def generate_postmortem(self, incident):
        """Generate postmortem report"""
        severity_names = {0: 'P0-Critical', 1: 'P1-High', 2: 'P2-Medium', 3: 'P3-Low'}
        
        report = f"""
{'='*60}
POSTMORTEM REPORT: {incident.id}
{'='*60}

Title: {incident.title}
Severity: {severity_names.get(incident.severity, 'Unknown')}
Status: {incident.status}

TIMELINE:
"""
        for ts, status, note in incident.timeline:
            report += f"  [{ts}] {status}: {note}\n"
        
        report += f"""
ROOT CAUSE:
  {incident.root_cause or 'Not identified'}

MITIGATION:
  {incident.mitigation or 'Not applied'}

ACTION ITEMS:
"""
        for i, item in enumerate(incident.action_items, 1):
            report += f"  {i}. [{item['priority']}] {item['item']} (Owner: {item['owner']})\n"
        
        report += f"\n{'='*60}"
        return report

# Demo incident response
print("=== Incident Response System ===\n")

irs = IncidentResponseSystem()

# Show available runbooks
print("Available Runbooks:")
for key, runbook in irs.runbooks.items():
    print(f"\n  📋 {runbook['title']}")
    for step in runbook['steps'][:3]:
        print(f"     {step}")
    print(f"     ... ({len(runbook['steps'])} steps total)")
    print(f"     Rollback: {runbook['rollback']}")

# Simulate an incident
print("\n" + "="*60)
print("SIMULATING INCIDENT: Model Performance Degradation")
print("="*60 + "\n")

incident = irs.create_incident(
    title="Recommendation model accuracy dropped 15%",
    severity=IncidentSeverity.P1_HIGH,
    description="Monitoring alert: model accuracy dropped from 85% to 72%"
)

print(f"Incident created: {incident.id}")
print(f"Status: {incident.status}")

# Follow runbook
runbook = irs.get_runbook('model_degradation')
print(f"\nFollowing runbook: {runbook['title']}")

# Simulate response steps
incident.update_status("investigating", "Checking monitoring dashboard")
incident.update_status("investigating", "Data drift detected: input distribution shifted")
incident.set_root_cause("Upstream data provider changed feature encoding format")
incident.set_mitigation("Rolled back to model v2.1.0, accuracy restored to 84%")
incident.resolve("Deployed data validation pipeline to catch format changes")

# Add action items
incident.add_action_item("Add data schema validation to ingestion pipeline", "Data Team", "P1")
incident.add_action_item("Set up automated drift detection alerts", "ML Team", "P1")
incident.add_action_item("Create data contract with upstream provider", "Platform Team", "P2")
incident.add_action_item("Add integration tests for data format changes", "QA Team", "P2")

# Generate postmortem
postmortem = irs.generate_postmortem(incident)
print(postmortem)

# Visualize incident metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simulated accuracy timeline during incident
hours = np.arange(0, 48)
accuracy = np.ones(48) * 85
# Incident starts at hour 12
accuracy[12:] = 72 + np.random.randn(36) * 2
# Mitigation at hour 18
accuracy[18:] = 84 + np.random.randn(30) * 1
# Full resolution at hour 24
accuracy[24:] = 85 + np.random.randn(24) * 0.5

axes[0].plot(hours, accuracy, 'b-', linewidth=2)
axes[0].axhline(y=85, color='green', linestyle='--', alpha=0.5, label='Target (85%)')
axes[0].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='Alert Threshold (80%)')
axes[0].axvspan(12, 18, alpha=0.2, color='red', label='Incident')
axes[0].axvspan(18, 24, alpha=0.2, color='yellow', label='Mitigation')
axes[0].set_xlabel('Hours')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Model Accuracy During Incident', fontsize=12, weight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Incident response time breakdown
phases = ['Detection', 'Triage', 'Investigation', 'Mitigation', 'Resolution']
times = [0.5, 0.5, 3.0, 2.0, 6.0]
colors = ['#F44336', '#FF9800', '#FFC107', '#4CAF50', '#2196F3']

axes[1].barh(phases, times, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Time (hours)')
axes[1].set_title('Incident Response Timeline', fontsize=12, weight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

for i, t in enumerate(times):
    axes[1].text(t + 0.1, i, f'{t}h', va='center')

plt.tight_layout()
plt.show()

print("事故响应核心原则：")
print("1. 先缓解，后调查（减少影响时间）")
print("2. 使用预定义Runbook加速响应")
print("3. 每次事故都要做复盘（Postmortem）")
print("4. 复盘重点：如何预防，而非追责")
print("5. Action Items必须有Owner和截止日期")
print("\n✓ 事故响应系统完成！")

### 8.2 常见问题

#### Q1: 模型多久需要重训练一次？

**A:**
- 取决于数据漂移速度
- 推荐：设置漂移检测自动触发
- 一般：每周/每月评估，按需重训练
- 关键：建立自动化重训练流水线

#### Q2: 如何处理模型更新导致的服务中断？

**A:**
- 使用蓝绿部署或灰度发布
- 保持旧版本热备，随时回滚
- 使用负载均衡器平滑切换
- 在低峰期执行更新

#### Q3: 生产环境中如何调试模型问题？

**A:**
- 结构化日志记录每个请求
- 保存异常输入样本
- 使用请求ID追踪完整链路
- 建立离线复现环境

#### Q4: 什么指标最重要？

**A:**
- 业务指标（转化率、用户满意度）> 模型指标（准确率）> 系统指标（延迟）
- 不同场景优先级不同
- 建议：定义3-5个核心指标，持续监控

#### Q5: 如何处理模型的公平性和偏见问题？

**A:**
- 部署前进行公平性审计
- 监控不同群体的模型表现
- 建立反馈机制收集用户投诉
- 定期审查和更新模型

### 8.3 本章总结

#### 核心要点

1. **MLOps基础**
   - 生产环境 ≠ 研究环境
   - 可复现性、可靠性、可维护性是核心
   - MLOps生命周期：开发→训练→验证→部署→监控→迭代

2. **模型版本管理**
   - 语义版本号：MAJOR.MINOR.PATCH
   - 追踪模型、数据、代码、超参数
   - 模型注册表管理staging/production状态

3. **A/B测试与灰度发布**
   - 统计显著性验证模型改进
   - 灰度发布：5%→25%→50%→100%
   - 自动回滚机制保障安全

4. **监控与漂移检测**
   - 监控性能指标、模型指标、数据分布
   - KS检验和PSI检测数据漂移
   - 分级告警：INFO→WARNING→CRITICAL

5. **错误处理与调试**
   - 输入验证、模型容错、系统保护
   - 优雅降级：备用模型、缓存结果
   - 结构化日志和请求追踪

6. **SLA与性能优化**
   - 定义可用性、延迟、吞吐量目标
   - 持续监控SLI，预警SLO违规
   - 缓存、批处理、模型优化

7. **事故响应**
   - 检测→分级→缓解→恢复→复盘
   - 预定义Runbook加速响应
   - 复盘驱动持续改进

#### 生产部署检查清单

```
部署前：
  □ 模型通过所有测试（单元、集成、性能）
  □ A/B测试验证改进显著
  □ 回滚方案已准备
  □ 监控和告警已配置
  □ 文档已更新

部署中：
  □ 灰度发布（5%→25%→50%→100%）
  □ 实时监控关键指标
  □ 准备随时回滚

部署后：
  □ 确认指标正常
  □ 监控数据漂移
  □ 定期评估模型性能
  □ 计划下次迭代
```

### 8.4 Module 7 总结

**Module 7 完整内容**：
1. ✅ 推理优化（量化、剪枝、蒸馏、ONNX、硬件加速）
2. ✅ 模型服务（REST API、批处理、负载均衡）
3. ✅ 生产最佳实践（版本管理、A/B测试、监控、事故响应）

### 💡 思考题

1. 如何设计一个自动化的模型重训练流水线？触发条件是什么？

2. A/B测试中，如何处理长尾效应和季节性变化？

3. 数据漂移和概念漂移的区别是什么？分别如何检测？

4. 在微服务架构中，如何管理多个模型的版本和依赖？

5. 如何平衡模型更新频率和系统稳定性？

## 思考题参考答案

### 问题 1：如何设计一个有效的 A/B 测试方案来评估新模型版本？

A/B 测试是科学评估模型版本优劣的核心手段，需要从**实验设计、流量分配、统计分析**三个维度系统规划。

**A/B 测试的核心原则：**

1. **单变量原则**：每次只改变一个变量（模型版本），控制其他所有条件一致
2. **随机分配**：使用一致性哈希确保用户稳定分配，避免用户在 A/B 组之间切换

**实验设计五步法：**

**第一步：明确主指标（Primary Metric）**

| 业务场景 | 主指标 | 辅助指标 |
|---------|--------|---------|
| 推荐系统 | 点击率（CTR） | 转化率、用户留存 |
| 文本分类 | 准确率/F1 | 延迟、用户反馈 |
| 对话系统 | 用户满意度 | 对话轮次、任务完成率 |

**第二步：计算所需样本量**

$$n = \frac{2 \cdot (z_{\alpha/2} + z_\beta)^2 \cdot \sigma^2}{\delta^2}$$

其中：
- $z_{\alpha/2} = 1.96$（显著性水平 $\alpha = 0.05$）
- $z_\beta = 0.84$（统计功效 80%）
- $\sigma^2$：指标方差（历史数据估计）
- $\delta$：最小可检测效应量（MDE，如期望 CTR 提升 1%）

**第三步：实验持续时间**

$$T = \frac{n}{\text{日均流量} \times \text{实验流量比例}}$$

通常建议最少 7 天（覆盖完整的周效应），最长不超过 4 周。

**第四步：流量分配策略**

```python
def assign_user_to_group(user_id: str, experiment_id: str, traffic_ratio: float = 0.1):
    """使用一致性哈希确保用户稳定分配"""
    import hashlib
    hash_val = int(hashlib.md5(f"{user_id}_{experiment_id}".encode()).hexdigest(), 16)
    bucket = hash_val % 100  # 0-99
    
    if bucket < traffic_ratio * 100:
        return "treatment"  # 新模型（B 组）
    else:
        return "control"    # 旧模型（A 组）
```

**第五步：统计检验**

对于比例型指标（CTR、准确率），使用 **Z 检验**：

$$z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$

若 $|z| > 1.96$，则在 95% 置信水平下差异显著。

**常见陷阱：**
- 新颖效应（Novelty Effect）：新版本初期因用户好奇而效果虚高
- 多重检验问题：同时测试多个指标需要 Bonferroni 校正 $\alpha' = \alpha / k$
- 过早停止（Peeking Problem）：不要在实验结束前根据中间结果决策

---

### 问题 2：如何检测和处理生产环境中的数据漂移？

数据漂移（Data Drift）是生产模型性能下降的最常见原因之一，包括**特征分布漂移**（Covariate Shift）和**标签漂移**（Concept Drift）。

**漂移的类型与特征：**

| 漂移类型 | 数学定义 | 现象 | 处理方法 |
|---------|---------|------|---------|
| 协变量漂移 | $P_{\text{train}}(X) \neq P_{\text{prod}}(X)$ | 输入分布变化，但映射关系不变 | 重新采样/加权训练 |
| 概念漂移 | $P_{\text{train}}(Y|X) \neq P_{\text{prod}}(Y|X)$ | 输入到输出的映射关系改变 | 必须重新训练 |
| 标签漂移 | $P_{\text{train}}(Y) \neq P_{\text{prod}}(Y)$ | 类别分布变化 | 调整分类阈值 |

**漂移检测方法：**

**1. 统计检验法（适合低维特征）**

```python
from scipy import stats

def detect_drift_ks_test(reference_data, current_data, threshold=0.05):
    """
    Kolmogorov-Smirnov 检验：检测连续特征分布是否发生变化
    """
    ks_stat, p_value = stats.ks_2samp(reference_data, current_data)
    
    drift_detected = p_value < threshold
    print(f"KS统计量: {ks_stat:.4f}, p值: {p_value:.4f}")
    print(f"漂移{'已' if drift_detected else '未'}检测到")
    return drift_detected
```

**2. PSI（Population Stability Index）**

$$\text{PSI} = \sum_{i=1}^{n} (A_i - E_i) \times \ln\left(\frac{A_i}{E_i}\right)$$

其中 $A_i$ 为当前分布，$E_i$ 为参考分布（按区间统计）。

| PSI 值 | 解释 | 操作建议 |
|--------|------|---------|
| < 0.1 | 无明显漂移 | 维持现状 |
| 0.1 - 0.2 | 轻微漂移 | 关注监控 |
| > 0.2 | 显著漂移 | 触发重训练 |

**3. 嵌入空间漂移检测（适合 NLP/CV）**

```python
def detect_embedding_drift(reference_embeddings, current_embeddings):
    """
    计算参考集和当前集的嵌入中心距离
    """
    ref_centroid = reference_embeddings.mean(axis=0)
    cur_centroid = current_embeddings.mean(axis=0)
    drift_score = np.linalg.norm(cur_centroid - ref_centroid)
    return drift_score
```

**生产级漂移监控流水线：**

```
每日数据收集
    ↓
特征统计计算（均值、方差、分位数）
    ↓
与参考分布对比（PSI/KS 检验）
    ↓
漂移分数 > 阈值？
    ├── 否 → 记录日志，继续监控
    └── 是 → 发送告警 → 人工审查
                 ↓
         漂移确认？
             ├── 否 → 调整阈值
             └── 是 → 触发重训练流水线
```

---

### 问题 3：SLA、SLO 和 SLI 之间有什么关系？如何制定合理的 SLA？

这三个概念构成了**服务可靠性的完整定义体系**，是 SRE（站点可靠性工程）的核心概念。

**三者定义与关系：**

$$\text{SLA（合同）} \supseteq \text{SLO（内部目标）} \supseteq \text{SLI（实际度量）}$$

| 概念 | 全称 | 定义 | 示例 |
|------|------|------|------|
| **SLI** | Service Level Indicator | 实际可观测的度量值 | 过去 1 小时 P99 延迟 = 185ms |
| **SLO** | Service Level Objective | 内部期望达到的目标 | P99 延迟 < 200ms，可用性 > 99.9% |
| **SLA** | Service Level Agreement | 对外承诺的合同保障 | 月可用性 ≥ 99.5%，违约赔偿 |

**关系图示：**
```
SLI（实测）        ≤    SLO（内部目标）   ≤    SLA（对外承诺）
  P99 = 185ms           P99 < 200ms            P99 < 500ms
  可用性 = 99.95%        可用性 > 99.9%          可用性 > 99.5%
```

内部 SLO 比 SLA 更严格，留出**错误预算（Error Budget）**缓冲空间。

**制定合理 SLA 的方法：**

**第一步：确定关键 SLI**

```python
# 推荐的 SLI 指标体系
sli_definitions = {
    "可用性": {
        "公式": "成功请求数 / 总请求数",
        "采集频率": "每分钟",
        "聚合周期": "30天滚动窗口"
    },
    "延迟": {
        "指标": ["P50", "P95", "P99"],
        "采集频率": "每秒",
        "聚合周期": "1小时"
    },
    "错误率": {
        "公式": "5xx 错误数 / 总请求数",
        "采集频率": "每分钟",
        "聚合周期": "1小时"
    }
}
```

**第二步：基于历史数据设置 SLO**

$$\text{推荐 SLO} = \text{历史第 5 百分位} \times 0.95$$

即取历史表现最差的 5% 情况作为上限，再留 5% 缓冲。

**第三步：错误预算管理**

$$\text{错误预算（月）} = (1 - \text{SLO}) \times 30 \times 24 \times 60 \text{ 分钟}$$

例如：SLO = 99.9%，则月错误预算 = 0.1% × 43200 = 43.2 分钟

当错误预算耗尽时，应暂停新功能发布，专注于可靠性改进。

---

### 问题 4：如何建立高效的事故响应流程？

有效的事故响应需要**标准化流程 + 清晰角色分工 + 事后学习机制**三者结合。

**事故生命周期：**

```
检测（Detection）→ 响应（Response）→ 缓解（Mitigation）→ 解决（Resolution）→ 复盘（Postmortem）
```

**事故严重等级分类：**

| 级别 | 业务影响 | 响应时间 | 示例 |
|------|---------|---------|------|
| P0（严重） | 核心功能完全不可用 | 5分钟内 | 推理服务全部宕机 |
| P1（高） | 核心功能严重降级 | 15分钟内 | 错误率 > 10% |
| P2（中） | 部分功能受损 | 1小时内 | 延迟升高 2 倍 |
| P3（低） | 轻微影响 | 1天内 | 单个地区偶发超时 |

**响应流程标准化（以 P1 为例）：**

```python
class IncidentResponse:
    """事故响应流程"""
    
    def detect(self, alert):
        """检测阶段（< 5 分钟）"""
        # 1. 告警触发
        # 2. 值班工程师确认
        # 3. 创建事故频道（Slack/企业微信）
        self.create_incident_channel(alert)
        self.notify_on_call_engineer()
    
    def diagnose(self):
        """诊断阶段（< 15 分钟）"""
        # 优先缓解，再诊断根因
        checks = [
            self.check_recent_deployments,  # 近期是否有变更？
            self.check_error_logs,           # 错误日志有何异常？
            self.check_upstream_services,    # 上游依赖是否正常？
            self.check_resource_utilization  # 资源是否耗尽？
        ]
        return [check() for check in checks]
    
    def mitigate(self, diagnosis):
        """缓解阶段（降低影响）"""
        if diagnosis.cause == "recent_deployment":
            self.rollback_deployment()       # 回滚
        elif diagnosis.cause == "traffic_spike":
            self.scale_out_replicas()        # 扩容
        elif diagnosis.cause == "upstream_failure":
            self.enable_fallback()           # 启用降级
    
    def postmortem(self, incident):
        """复盘（事故后 48 小时内）"""
        return {
            "timeline": incident.timeline,
            "root_cause": incident.root_cause,
            "impact": incident.user_impact,
            "action_items": [
                # 具体、可执行、有负责人和截止日期
                {"action": "添加内存告警", "owner": "张工", "due": "2周内"},
                {"action": "完善降级策略", "owner": "李工", "due": "1个月内"}
            ]
        }
```

**无指责文化（Blameless Postmortem）原则：**

- 关注**系统性问题**，而非个人失误
- 探讨"**为什么系统允许此类错误发生**"，而非"谁犯了错"
- 每次事故都是改进系统的机会

---

### 问题 5：如何在保证服务可用性的同时推进持续的模型迭代？

这是 MLOps 的核心矛盾：**稳定性（不破坏现有服务）vs 迭代速度（快速上线新能力）**。

**核心解决思路：将"部署"与"发布"解耦**

```
模型训练 → CI 验证 → 部署到生产（流量=0%）→ 灰度发布 → 全量发布
                ↑                                    ↑
           （自动化）                           （人工审批）
```

**持续集成（CI）流水线设计：**

```yaml
# CI 检查清单（每次模型提交自动触发）
stages:
  - name: 代码质量
    checks:
      - lint（代码规范）
      - type_check（类型检查）
      - unit_tests（单元测试）
  
  - name: 模型质量
    checks:
      - accuracy_regression（准确率不低于基线 0.5%）
      - latency_benchmark（P99 延迟 < SLO 阈值）
      - memory_usage（内存占用 < 上限）
      - data_drift_check（离线测试集分布验证）
  
  - name: 安全性
    checks:
      - adversarial_robustness（对抗样本测试）
      - bias_audit（公平性审查）
      - pii_leakage（隐私信息泄漏检测）
```

**Feature Flag 与流量控制：**

```python
class ModelRouter:
    """通过 Feature Flag 控制模型版本流量"""
    
    def __init__(self, config_service):
        self.config = config_service
    
    def route(self, user_id: str, request):
        # 从配置服务动态读取流量分配（可实时修改，无需重启）
        canary_traffic = self.config.get("model_v2_traffic", default=0.0)
        
        if self.is_canary_user(user_id, canary_traffic):
            return self.model_v2.predict(request)
        else:
            return self.model_v1.predict(request)
    
    def is_canary_user(self, user_id: str, ratio: float) -> bool:
        bucket = hash(user_id) % 100
        return bucket < ratio * 100
```

**模型迭代速度 vs 稳定性的平衡矩阵：**

| 模型类型 | 更新频率 | 灰度时间 | 自动化程度 |
|---------|---------|---------|----------|
| 核心推理模型 | 月级别 | 1-2 周 | 人工审批 |
| 辅助分类模型 | 周级别 | 3-5 天 | 半自动 |
| 规则/过滤模型 | 日级别 | 1 天 | 全自动（满足指标则全量） |

**错误预算驱动决策：**

```
错误预算剩余 > 50%：可以大胆迭代，快速发布
错误预算剩余 20%-50%：谨慎迭代，加强测试
错误预算剩余 < 20%：停止新功能发布，专注稳定性
错误预算耗尽：冻结所有变更，全力修复
```